In [21]:
import pandas as pd
from pyprojroot import here
from scipy import stats
from statsmodels.stats.multitest import multipletests
import numpy as np
import json

In [22]:
RESULTS_DATA_DIR = here() / "results"

In [23]:
df = pd.read_csv(RESULTS_DATA_DIR/"predictions_benchmark.csv")

### We perform a Wilcoxon test with Benjamini-Hochberg correction using Markov results as a comparison base and the models error.

In [24]:
models = predictions.columns.tolist()[2:9]

markov   = df['markov'].values.astype(float)
observed = df['observed'].values.astype(float)

In [25]:
results_pred = []
for model in models:
    pred = df[model].values.astype(float)
    mask = ~(np.isnan(markov) | np.isnan(pred))
    m_clean, p_clean = markov[mask], pred[mask]

    result   = stats.wilcoxon(m_clean, p_clean, alternative='two-sided', nan_policy='omit')
    med_diff = float(np.nanmedian(p_clean - m_clean))

    results_pred.append({
        'model':       model,
        'n':           int(mask.sum()),
        'W':           float(result.statistic),
        'p_raw':       float(result.pvalue),
        'median_diff': round(med_diff, 4),   # positive = model predicts higher than Markov
    })

pred_df = pd.DataFrame(results_pred)
reject, p_adj, _, _ = multipletests(pred_df['p_raw'], method='fdr_bh')
pred_df['p_bh']        = [float(x) for x in p_adj]
pred_df['significant'] = [bool(x)  for x in reject]

print('=== Table 1: Predicted values vs. Markov ===')
print(pred_df.to_string(index=False))

=== Table 1: Predicted values vs. Markov ===
       model    n         W        p_raw  median_diff         p_bh  significant
      hybrid 2456   14410.0 2.781152e-02        0.000 3.244677e-02         True
markov_fuzzy 2456  506971.0 1.354819e-31        0.000 9.483736e-31         True
         ols 2456 1131304.0 6.841831e-27        1.125 2.394641e-26         True
         wma 2456 1356369.5 1.983559e-05        0.825 2.776982e-05         True
          rd 2456 1296154.0 2.670168e-09        1.170 4.672793e-09         True
       arima 2456 1477228.0 4.298620e-01       -0.080 4.298620e-01        False
 persistence 2456 1260206.5 1.709702e-12       -0.080 3.989305e-12         True


In [26]:
markov_ae = np.abs(markov - observed)

results_err = []
for model in models:
    pred     = df[model].values.astype(float)
    model_ae = np.abs(pred - observed)
    mask     = ~(np.isnan(markov_ae) | np.isnan(model_ae))
    m_clean, p_clean = markov_ae[mask], model_ae[mask]

    result   = stats.wilcoxon(m_clean, p_clean, alternative='two-sided', nan_policy='omit')
    med_diff = float(np.nanmedian(p_clean - m_clean))

    results_err.append({
        'model':       model,
        'n':           int(mask.sum()),
        'W':           float(result.statistic),
        'p_raw':       float(result.pvalue),
        'median_diff': round(med_diff, 4),   # negative = model has lower AE than Markov (better)
    })

err_df = pd.DataFrame(results_err)
reject, p_adj, _, _ = multipletests(err_df['p_raw'], method='fdr_bh')
err_df['p_bh']        = [float(x) for x in p_adj]
err_df['significant'] = [bool(x)  for x in reject]

print('\n=== Table 2: Absolute errors vs. Markov (lower = better) ===')
print(err_df.to_string(index=False))



=== Table 2: Absolute errors vs. Markov (lower = better) ===
       model    n         W        p_raw  median_diff         p_bh  significant
      hybrid 2456   15508.5 1.935828e-01         0.00 2.258466e-01        False
markov_fuzzy 2456  713103.5 7.324939e-02         0.00 1.281864e-01        False
         ols 2456 1450528.0 9.846020e-02        -0.08 1.378443e-01        False
         wma 2456 1493565.0 7.201156e-01        -0.08 7.201156e-01        False
          rd 2456  932814.5 8.730648e-60         2.72 6.111454e-59         True
       arima 2456 1438716.5 5.912681e-02        -0.08 1.281864e-01        False
 persistence 2456 1269612.0 1.249964e-11        -0.08 4.374874e-11         True


In [27]:
pred_df.to_csv(RESULTS_DATA_DIR/'wilcoxon_vs_markov_predictions.csv', index=False)
err_df.to_csv(RESULTS_DATA_DIR/'wilcoxon_vs_markov_errors.csv', index=False)
print('\nSaved.')


Saved.
